In [0]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName('pratices').getOrCreate()


In [0]:
df=spark.read.csv('/Volumes/practical/data/files/silver/test2.csv',header=True,inferSchema=True)

In [0]:
display(df)

In [0]:
#dropping the coloumn
df_drop=df.drop('Name')
df_drop.display()

In [0]:
display(df)

In [0]:
#null values 
df_dr=df.na.drop(how='any',thresh=2)
df_dr.display()

In [0]:
df_dr1=df.na.drop(how='any',subset=['Name','Salary'])
df_dr1.display()

In [0]:

df_dr3 = df.fillna({
    'name': 'unknown',
    'Experience': 0,
    'age': 0
})
display(df_dr3)

In [0]:
from pyspark.ml.feature import Imputer
imputer = Imputer(
    inputCols=['age', 'Experience', 'Salary'],
    outputCols=["{}_imputed".format(c) for c in ['age', 'Experience', 'Salary']]
).setStrategy("mean")
df.withColumn('age',df['age'].cast('int'))

In [0]:
from pyspark.sql.functions import col
df = df.withColumn('age', col('age').cast('integer'))


In [0]:
imputer.fit(df).transform(df).show()

In [0]:
df1=spark.read.csv('/Volumes/practical/data/files/silver/test1.csv',header=True,inferSchema=True)
display(df1)

filter operation


In [0]:
df.filter('salary<20000').show()

In [0]:
#~ not equalls in pyspark
df.filter(~(df['salary']<20000)).show()
 

In [0]:
df3=spark.read.csv('/Volumes/practical/data/files/silver/test3.csv',header=True,inferSchema=True)
display(df3)


Group By and aggeration 

In [0]:
df3.groupBy('name').sum().show()

In [0]:
df3.groupBy('departments').sum().show()

In [0]:
data = [
 (1,"A"), (2,"B"), (1,"A"), (3,"C"), (2,"B")
]
cols = ["id","name"]
df = spark.createDataFrame(data, cols)
display(df)

In [0]:
df.distinct().show()

In [0]:
data = [
 (1,"IT",60000),
 (2,"IT",80000),
 (3,"IT",70000),
 (4,"HR",50000),
 (5,"HR",60000)
]
cols = ["emp_id","dept","salary"]
df1 = spark.createDataFrame(data, cols)
display(df1)

In [0]:
df1.createOrReplaceTempView('employee')

In [0]:
from pyspark.sql import Window
from pyspark.sql.functions import *
w = Window.partitionBy('dept').orderBy(col('salary').desc())
df1.withColumn('rank',dense_rank().over(w)).show()


In [0]:
from pyspark.sql import Window
from pyspark.sql.functions import *
w = Window.partitionBy('dept').orderBy(col('salary').desc())
df1.withColumn('rank',dense_rank().over(w)).filter(col('rank')==2).show()

In [0]:
%sql
select * from employee

In [0]:
%sql
with cte as(select *,dense_rank() over(partition by dept order by salary desc) as rn from employee)select * from cte where rn=2

In [0]:
data = [
 (101,"2024-01-01"),
 (101,"2024-01-01"),
 (102,"2024-01-02"),
 (101,"2024-01-02")
]
cols = ["customer_id","order_date"]
df4 = spark.createDataFrame(data, cols)
display(df4)

In [0]:
from pyspark.sql.functions import *
df4.groupBy('order_date').agg(count('customer_id').alias('num_orders')).show()

In [0]:
df_sql=df4.createOrReplaceTempView('orders')
display(df_sql)

In [0]:
%sql
select order_date,count(customer_id)as ordered_no from employee group by order_date 